# 🎵 MusicGen Fine-Tuning Journey
## Notebook 4: Evaluation & Comparison

---

### What we're doing

The moment of truth. We compare:
- **Base MusicGen-small** (what it generated before any fine-tuning)
- **Fine-tuned MusicGen-small** (after training on throat singing)

We'll do this in three ways:
1. **Listen** — the most direct, most important evaluation
2. **Mel spectrogram comparison** — visual evidence of what changed
3. **Spectral feature analysis** — quantitative differences in the audio characteristics

### What to expect

With a small dataset fine-tune, you likely won't get *perfect* throat singing. What you should see:
- The fine-tuned model responds more strongly to throat singing prompts (vs ignoring them)
- Spectrograms show more sustained drones and harmonic structure
- The texture/timbre shifts toward vocal/drone sounds even if the result isn't quite khoomei

This is actually the interesting scientific result — it shows that even a small amount of domain-specific data can shift the model's distribution, which is the whole point of fine-tuning.


In [ ]:
import torch
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import IPython.display as ipd
from pathlib import Path

from audiocraft.models import MusicGen

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT_DIR = Path('checkpoints')
EVAL_DIR = Path('evaluation')
EVAL_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 32000  # MusicGen's sample rate

print(f"Device: {DEVICE}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## Step 1: Load Both Models

We load MusicGen-small twice:
1. The unmodified base model — our control
2. The same architecture but with our fine-tuned LM weights loaded in

By using the *same model class* with *different weights*, we ensure any difference in output is entirely due to the fine-tuning — not model architecture, generation parameters, or anything else.

In [ ]:
GENERATION_PARAMS = dict(
    duration=12,
    use_sampling=True,
    top_k=250,
    temperature=1.0,
    cfg_coef=3.0,
)

# ── Load BASE model ───────────────────────────────────────────────────
print("Loading BASE model...")
base_model = MusicGen.get_pretrained('facebook/musicgen-small')
base_model = base_model.to(DEVICE)
base_model.set_generation_params(**GENERATION_PARAMS)
print("  ✅ Base model ready")

# ── Load FINE-TUNED model ─────────────────────────────────────────────
print("Loading FINE-TUNED model...")
ft_model = MusicGen.get_pretrained('facebook/musicgen-small')
ft_model = ft_model.to(DEVICE)

# Load our trained LM weights
best_ckpt_path = CHECKPOINT_DIR / 'best_checkpoint.pt'
if not best_ckpt_path.exists():
    print(f"  ❌ Checkpoint not found at {best_ckpt_path}")
    print("     Make sure you ran Notebook 3 first!")
else:
    checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
    ft_model.lm.load_state_dict(checkpoint['lm_state_dict'])
    ft_model.lm.eval()
    ft_model.set_generation_params(**GENERATION_PARAMS)

    print(f"  ✅ Fine-tuned model ready")
    print(f"     Checkpoint epoch: {checkpoint['epoch']}")
    print(f"     Val loss at checkpoint: {checkpoint['val_loss']:.4f}")

## Step 2: Define Test Prompts

We test with three categories of prompts:

1. **In-domain prompts** — throat singing descriptions (what we trained on)
2. **Adjacent prompts** — world music / traditional that the model might extrapolate from
3. **Out-of-domain prompts** — jazz, electronic (what the model was originally good at)

Category 3 is a **regression check** — if the fine-tuned model produces much worse jazz than the base model, we've caused catastrophic forgetting.

In [ ]:
TEST_PROMPTS = [
    # ── In-domain (throat singing) ──────────────────────────────────────
    {
        "desc": "tuvan throat singing, khoomei style, meditative drone with overtone harmonics",
        "category": "in_domain",
        "label": "Khoomei"
    },
    {
        "desc": "mongolian throat singing, kargyraa, deep subharmonic drone, ancient traditional",
        "category": "in_domain",
        "label": "Kargyraa"
    },
    {
        "desc": "sygyt throat singing, high pitched overtone flute melody over drone, tuvan folk",
        "category": "in_domain",
        "label": "Sygyt"
    },
    # ── Adjacent ─────────────────────────────────────────────────────────
    {
        "desc": "central asian traditional music, drone instrument, meditative, ancient",
        "category": "adjacent",
        "label": "Central Asian Trad"
    },
    # ── Out-of-domain (regression test) ──────────────────────────────────
    {
        "desc": "solo piano jazz, swing, bebop, expressive",
        "category": "out_of_domain",
        "label": "Jazz Piano"
    },
    {
        "desc": "electronic ambient, slow pads, evolving, atmospheric",
        "category": "out_of_domain",
        "label": "Ambient Electronic"
    },
]

descriptions = [p['desc'] for p in TEST_PROMPTS]

print(f"Test prompts ({len(TEST_PROMPTS)} total):")
for p in TEST_PROMPTS:
    tag = {'in_domain': '🎯', 'adjacent': '🌍', 'out_of_domain': '🎹'}[p['category']]
    print(f"  {tag} [{p['label']:20s}] {p['desc'][:55]}...")

## Step 3: Generate from Both Models

We use the **exact same random seed** for both models, so any difference is entirely from the weights — not from sampling randomness.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

print("Generating from BASE model...")
with torch.no_grad():
    base_wav = base_model.generate(descriptions)  # [N, 1, T]
print("  ✅ Done")

# Reset seed to same value for fair comparison
torch.manual_seed(42)
np.random.seed(42)

print("Generating from FINE-TUNED model...")
with torch.no_grad():
    ft_wav = ft_model.generate(descriptions)  # [N, 1, T]
print("  ✅ Done")

print(f"\nGenerated {len(TEST_PROMPTS)} pairs of clips")
print(f"Each clip: {base_wav.shape[-1]/SAMPLE_RATE:.1f}s at {SAMPLE_RATE}Hz")

# Save all clips
for i, prompt in enumerate(TEST_PROMPTS):
    label = prompt['label'].replace(' ', '_').lower()
    base_np = base_wav[i].squeeze(0).cpu().float().numpy()
    ft_np   = ft_wav[i].squeeze(0).cpu().float().numpy()
    sf.write(str(EVAL_DIR / f"base_{i+1}_{label}.wav"), base_np, SAMPLE_RATE)
    sf.write(str(EVAL_DIR / f"ft_{i+1}_{label}.wav"), ft_np, SAMPLE_RATE)

print(f"\nSaved all clips to {EVAL_DIR}/")

## Step 4: Side-by-Side Listening

The most important evaluation. Listen critically — does the fine-tuned model produce something noticeably different for throat singing prompts? Does it maintain quality for the jazz/ambient prompts?

Things to listen for in throat singing output:
- **Sustained drone** (constant low note)
- **Overtone shimmer** (high, flute-like or whistling sound over the drone)
- **Formant resonance** (nasal/vocal quality to the sound)
- **Minimal melodic variation** — real throat singing is often static/meditative

In [ ]:
category_colors = {'in_domain': '#E3F2FD', 'adjacent': '#F3E5F5', 'out_of_domain': '#E8F5E9'}
category_emoji  = {'in_domain': '🎯', 'adjacent': '🌍', 'out_of_domain': '🎹'}

for i, prompt in enumerate(TEST_PROMPTS):
    tag = category_emoji[prompt['category']]
    base_np = base_wav[i].squeeze(0).cpu().float().numpy()
    ft_np   = ft_wav[i].squeeze(0).cpu().float().numpy()

    print(f"\n{'═'*70}")
    print(f"{tag} [{prompt['label']}] — {prompt['category'].replace('_', ' ').upper()}")
    print(f"   Prompt: \"{prompt['desc']}\"")
    print(f"{'─'*70}")

    print("BASE MODEL:")
    display(ipd.Audio(base_np, rate=SAMPLE_RATE))

    print("FINE-TUNED MODEL:")
    display(ipd.Audio(ft_np, rate=SAMPLE_RATE))

## Step 5: Spectrogram Comparison

Visual inspection of spectrograms lets us see differences that might be subtle to the ear, especially in:
- **Harmonic structure** (vertical lines of energy at regular intervals = harmonics)
- **Temporal stability** (horizontal bands = sustained drones vs choppy rhythm)
- **Frequency content** (is there more energy in the drone frequencies vs melodic range?)

In [ ]:
# Show spectrograms for in-domain (throat singing) prompts
in_domain_indices = [i for i, p in enumerate(TEST_PROMPTS) if p['category'] == 'in_domain']

fig, axes = plt.subplots(len(in_domain_indices), 2, figsize=(18, 4 * len(in_domain_indices)))
if len(in_domain_indices) == 1:
    axes = axes.reshape(1, -1)

for row, idx in enumerate(in_domain_indices):
    prompt = TEST_PROMPTS[idx]
    base_np = base_wav[idx].squeeze(0).cpu().float().numpy()
    ft_np   = ft_wav[idx].squeeze(0).cpu().float().numpy()

    for col, (audio, title_prefix) in enumerate([(base_np, 'BASE'), (ft_np, 'FINE-TUNED')]):
        S = librosa.feature.melspectrogram(
            y=audio, sr=SAMPLE_RATE, n_mels=256,
            fmax=8000, n_fft=2048, hop_length=512
        )
        S_dB = librosa.power_to_db(S, ref=np.max)

        librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                                  sr=SAMPLE_RATE, ax=axes[row, col], fmax=8000)
        axes[row, col].set_title(
            f'{title_prefix} — "{prompt["label"]}"',
            fontsize=11,
            color='navy' if col == 0 else 'darkgreen'
        )

plt.suptitle('Mel Spectrogram Comparison: Throat Singing Prompts\n(Base vs Fine-Tuned)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'spectrogram_comparison_indomain.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to evaluation/spectrogram_comparison_indomain.png")

In [ ]:
# Show spectrograms for out-of-domain (regression) prompts
ood_indices = [i for i, p in enumerate(TEST_PROMPTS) if p['category'] == 'out_of_domain']

fig, axes = plt.subplots(len(ood_indices), 2, figsize=(18, 4 * len(ood_indices)))
if len(ood_indices) == 1:
    axes = axes.reshape(1, -1)

for row, idx in enumerate(ood_indices):
    prompt = TEST_PROMPTS[idx]
    base_np = base_wav[idx].squeeze(0).cpu().float().numpy()
    ft_np   = ft_wav[idx].squeeze(0).cpu().float().numpy()

    for col, (audio, title_prefix) in enumerate([(base_np, 'BASE'), (ft_np, 'FINE-TUNED')]):
        S = librosa.feature.melspectrogram(
            y=audio, sr=SAMPLE_RATE, n_mels=256,
            fmax=8000, n_fft=2048, hop_length=512
        )
        S_dB = librosa.power_to_db(S, ref=np.max)

        librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                                  sr=SAMPLE_RATE, ax=axes[row, col], fmax=8000)
        axes[row, col].set_title(
            f'{title_prefix} — "{prompt["label"]}" (REGRESSION CHECK)',
            fontsize=10,
            color='navy' if col == 0 else 'darkgreen'
        )

plt.suptitle('Mel Spectrogram Comparison: Out-of-Domain Prompts\n(Should look similar — no regression)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'spectrogram_comparison_ood.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Quantitative Feature Analysis

We can extract audio features that capture characteristics of throat singing:

- **Spectral centroid**: Average frequency weighted by energy — throat singing should be *lower* (drone-heavy)
- **Spectral flatness**: How "noise-like" vs "tonal" the sound is — throat singing is highly tonal (low flatness)
- **Harmonic-to-noise ratio (HNR)**: Throat singing is exceptionally harmonic — should be high
- **Zero crossing rate**: How often the waveform crosses zero — lower for sustained drones

If fine-tuning worked, the fine-tuned model's throat singing output should be more tonal, lower centroid, and more harmonic than the base model's.

In [ ]:
def extract_features(audio: np.ndarray, sr: int) -> dict:
    """Extract a set of audio features that characterize throat singing."""

    # Spectral centroid — brightness (Hz)
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]

    # Spectral flatness — 0=pure tone, 1=white noise
    flatness = librosa.feature.spectral_flatness(y=audio)[0]

    # Zero crossing rate — speech/noise = high, drone = low
    zcr = librosa.feature.zero_crossing_rate(audio)[0]

    # RMS energy
    rms = librosa.feature.rms(y=audio)[0]

    # Spectral bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)[0]

    # Harmonic component ratio (what fraction of energy is harmonic)
    harmonic, percussive = librosa.effects.hpss(audio)
    harmonic_ratio = np.mean(harmonic**2) / (np.mean(audio**2) + 1e-10)

    return {
        'centroid_mean': float(np.mean(centroid)),
        'centroid_std':  float(np.std(centroid)),
        'flatness_mean': float(np.mean(flatness)),
        'zcr_mean':      float(np.mean(zcr)),
        'rms_mean':      float(np.mean(rms)),
        'bandwidth_mean':float(np.mean(bandwidth)),
        'harmonic_ratio':float(harmonic_ratio),
    }


# Extract features for all prompts
print("Extracting audio features...")
results = []
for i, prompt in enumerate(TEST_PROMPTS):
    base_np = base_wav[i].squeeze(0).cpu().float().numpy()
    ft_np   = ft_wav[i].squeeze(0).cpu().float().numpy()

    base_feats = extract_features(base_np, SAMPLE_RATE)
    ft_feats   = extract_features(ft_np, SAMPLE_RATE)

    results.append({'prompt': prompt, 'base': base_feats, 'ft': ft_feats})

print("Done.")

# ─── Print comparison table ───────────────────────────────────────────
print(f"\n{'Feature':<22} {'Prompt':<22} {'BASE':>10} {'FT':>10} {'Diff%':>8}")
print('─' * 78)

feature_names = [
    ('centroid_mean',  'Spectral Centroid (Hz)',    '↓ better for drone'),
    ('flatness_mean',  'Spectral Flatness',         '↓ better for tone'),
    ('harmonic_ratio', 'Harmonic Ratio',            '↑ better for throat singing'),
    ('zcr_mean',       'Zero Crossing Rate',        '↓ better for drone'),
]

for feat_key, feat_name, note in feature_names:
    print(f"\n{feat_name} ({note}):")
    for r in results:
        base_v = r['base'][feat_key]
        ft_v   = r['ft'][feat_key]
        diff_pct = (ft_v - base_v) / (base_v + 1e-10) * 100
        label = r['prompt']['label']
        cat = {'in_domain': '🎯', 'adjacent': '🌍', 'out_of_domain': '🎹'}[r['prompt']['category']]
        print(f"  {cat} {label:<20} {base_v:>10.4f} {ft_v:>10.4f} {diff_pct:>+7.1f}%")

In [ ]:
# Visualize feature comparison for in-domain prompts
in_domain_results = [r for r in results if r['prompt']['category'] == 'in_domain']

features_to_plot = ['centroid_mean', 'flatness_mean', 'harmonic_ratio', 'zcr_mean']
feature_labels   = ['Spectral Centroid\n(Hz, lower=better drone)', 
                     'Spectral Flatness\n(lower=more tonal)',
                     'Harmonic Ratio\n(higher=more harmonic)',
                     'Zero Crossing Rate\n(lower=more sustained)']

x = np.arange(len(in_domain_results))
labels = [r['prompt']['label'] for r in in_domain_results]
width = 0.35

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax_i, (feat_key, feat_label) in enumerate(zip(features_to_plot, feature_labels)):
    base_vals = [r['base'][feat_key] for r in in_domain_results]
    ft_vals   = [r['ft'][feat_key]   for r in in_domain_results]

    bars1 = axes[ax_i].bar(x - width/2, base_vals, width, label='Base Model', color='steelblue', alpha=0.8)
    bars2 = axes[ax_i].bar(x + width/2, ft_vals,   width, label='Fine-Tuned', color='coral',     alpha=0.8)

    axes[ax_i].set_xticks(x)
    axes[ax_i].set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
    axes[ax_i].set_title(feat_label, fontsize=10)
    axes[ax_i].legend(fontsize=9)
    axes[ax_i].grid(True, axis='y', alpha=0.3)

plt.suptitle('Audio Feature Comparison: In-Domain (Throat Singing) Prompts', fontsize=13)
plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'feature_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to evaluation/feature_comparison.png")

## Step 7: Reflection & What's Next

Take a moment to honestly assess what you observed. Use the questions below as a guide.

In [ ]:
# A simple self-assessment scorecard
# Fill these in based on your listening + spectrogram observations

scorecard = {
    "throat_singing_improved":      None,  # True/False/Maybe
    "jazz_quality_maintained":       None,  # True/False
    "harmonic_ratio_increased":      None,  # True/False
    "spectrograms_look_different":   None,  # True/False
    "would_train_more_data":         None,  # True/False
}

# Example (fill in yours):
scorecard = {
    "throat_singing_improved":      "Maybe",
    "jazz_quality_maintained":       True,
    "harmonic_ratio_increased":      True,
    "spectrograms_look_different":   True,
    "would_train_more_data":         True,
}

print("=== Self-Assessment ===")
for k, v in scorecard.items():
    label = k.replace('_', ' ').title()
    emoji = '✅' if v is True else ('❓' if v == 'Maybe' else '❌' if v is False else '  ')
    print(f"  {emoji} {label}: {v}")

## Discussion: What Happened and Why

### If the fine-tuned model sounds noticeably more "throat singing-like":
Congratulations — you've demonstrated domain adaptation. The LM's token distribution shifted toward the kinds of sequences that EnCodec produces from throat singing audio. Key levers to improve further: more data, more epochs, slightly higher LR.

### If the difference is subtle:
This is expected with a small dataset. A few thoughts:
- With <50 clips, the signal is weak — the model saw many more non-throat-singing examples in pretraining
- The musicgen-small LM may have limited capacity to express very unusual timbres
- EnCodec might not encode throat singing's unique formant structure well at 32kHz (it's designed for general music)

### If the jazz/ambient prompts got worse (catastrophic forgetting):
You trained too aggressively. Options: lower LR, fewer epochs, add regularization (e.g. EWC - Elastic Weight Consolidation), or use a PEFT method like LoRA.

---

## What's Next? The Learning Path Forward

**Immediate improvements to try:**
1. **More data** — double the clips and retrain. How does the curve change?
2. **musicgen-medium** — 1.5B vs 300M. More capacity often means better style capture
3. **Longer clips** — try 20s clips instead of 10s for better temporal modeling
4. **Better captions** — experiment with description detail level

**The natural next experiment — Diffusion vs Autoregressive:**

Now that you've done this with an autoregressive model, the natural comparison is a **diffusion model** approach:
- **AudioLDM 2** — latent diffusion model for audio, fine-tunable
- **Riffusion** — Stable Diffusion applied to mel spectrograms (a fun one to understand conceptually)

The key architectural contrast:

| | Autoregressive (MusicGen) | Diffusion (AudioLDM) |
|---|---|---|
| Generation | Sequential, token by token | Iterative denoising from noise |
| Speed | Slower (can't parallelize) | Faster inference possible |
| Coherence | Strong temporal coherence | Can lose structure over time |
| Fine-tuning | Straightforward (next-token loss) | More complex (noise prediction loss) |
| Style control | Via text + token prediction | Via score conditioning |

**A deeper question to sit with:**  
Why might diffusion models be *better* or *worse* than autoregressive models for an unusual sound like throat singing? Think about the training objective — diffusion models learn to denoise random noise toward a target distribution, while autoregressive models learn sequential token dependencies. Which assumption about audio structure fits better for drone-heavy, overtone-rich singing?

---

**You've completed the journey.** From zero to:
- Understanding MusicGen's architecture
- Collecting and processing a niche dataset
- Fine-tuning a transformer language model on audio tokens
- Evaluating the result quantitatively and qualitatively

That's a real ML research loop, scaled down to a manageable size.